# NFL Fantasy Monte Carlo Simulator — Parameter Exploration

End-to-end validation of the statistical pipeline.

> **Data sources:** 2022–2024 pulled via `nfl_data_py`; 2025 pulled via `nflreadpy` (nflverse). Both are cached locally as parquet after the first run.

1. **Data pull** — 2022–2025 weekly stats  
2. **Fit on 2022–2024** — training split; estimate α, β, π per player  
3. **Player spot-checks** — do well-known players have sensible parameters?  
4. **Gamma fit visualization** — does the fitted distribution match actual scores?  
5. **Zero-inflation by position** — how often does each position score 0?  
6. **2025 holdout validation** — do 2022–2024 parameters predict 2025 well?  
7. **Refit on 2022–2025** — production parameters for the 2026 season  
8. **Simulation sanity** — single-team score distribution shape  
9. **Matchup demo** — head-to-head win probability with score distribution overlap  

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")   # non-interactive backend for headless execution
import matplotlib.pyplot as plt
from scipy.stats import gamma as scipy_gamma

from data import pull_weekly_stats
from fit import fit_player_params
from simulate import simulate_team_score
from matchup import simulate_matchup

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 120, "axes.titlesize": 11, "axes.labelsize": 10})

POS_COLORS = {"QB": "royalblue", "RB": "forestgreen", "WR": "darkorange", "TE": "mediumpurple"}
SKILL_POS  = ["QB", "RB", "WR", "TE"]


def top_n_at_pos(params: pd.DataFrame, pos: str, n: int) -> list[dict]:
    """Return top-n players at pos by expected PPR score as roster dicts."""
    return (
        params[(params["position"] == pos) & ~params["fallback_used"]]
        .nlargest(n, "exp_score")
        [["player_id", "player_name", "position"]]
        .to_dict("records")
    )


print("Setup complete.")

## 1. Data Pull (2022–2025)

2022–2024 is the **training split** used to estimate parameters. 2025 is held out for out-of-sample validation. After validation we refit on all four years to get the best available estimates for the 2026 season.

In [ ]:
weekly_all   = pull_weekly_stats([2022, 2023, 2024, 2025])
weekly_train = weekly_all[weekly_all["season"].isin([2022, 2023, 2024])].copy()
weekly_hold  = weekly_all[weekly_all["season"] == 2025].copy()

for label, df in [
    ("Train (2022-24)", weekly_train),
    ("Hold  (2025)   ", weekly_hold),
    ("All   (2022-25)", weekly_all),
]:
    print(
        f"{label}:  {len(df):>7,} player-weeks  "
        f"|  {df['player_id'].nunique():>4,} unique players  "
        f"|  seasons {sorted(df['season'].unique())}"
    )

## 2. Fit on 2022–2024 (Training Split)

**Method of moments** on non-zero weeks only:

$$\hat{\beta} = \frac{\hat{\sigma}^2}{\hat{\mu}} \qquad \hat{\alpha} = \frac{\hat{\mu}^2}{\hat{\sigma}^2} \qquad \hat{\pi} = 1 - \frac{n_{\text{nonzero}}}{17 \times n_{\text{seasons}}}$$

Players with fewer than 4 non-zero weeks (or σ = 0) fall back to positional-median parameters.  We pass `cache=False` to avoid overwriting the production cache before validation.

In [ ]:
params_train = fit_player_params(weekly_train, cache=False)
params_train["exp_score"] = (1 - params_train["pi"]) * params_train["mu"]

n_total = len(params_train)
n_fb    = params_train["fallback_used"].sum()
print(f"Total players: {n_total:,}  |  Fallback used: {n_fb:,} ({100*n_fb/n_total:.1f}%)\n")

fallback_by_pos = (
    params_train[params_train["position"].isin(SKILL_POS)]
    .groupby("position")["fallback_used"]
    .agg(n="count", n_fallback="sum")
    .assign(fallback_pct=lambda d: (100 * d["n_fallback"] / d["n"]).round(1))
    .sort_values("fallback_pct", ascending=False)
)
print("Fallback rate by skill position (2022-2024 fit):")
print(fallback_by_pos.to_string())

## 3. Player Spot-Checks

Top 3 players at each skill position by **expected PPR score per game** = (1 − π) × μ.

**Reading the table:**
- **α** (shape): higher → tighter / more consistent week-to-week. α < 2 → heavy right tail (boom-or-bust).
- **β** (scale): higher → bigger typical game when active. E[X | active] = αβ = μ.
- **π**: probability of exactly 0 production in a given week (injury, DNP, etc.).
- **exp_score**: unconditional expected score = (1 − π) × μ.

In [ ]:
rows = []
for pos in SKILL_POS:
    rows.append(
        params_train[params_train["position"] == pos]
        .nlargest(3, "exp_score")
        [["player_name", "position", "alpha", "beta", "pi", "mu", "exp_score", "n_nonzero"]]
    )

pd.concat(rows).reset_index(drop=True).round(3)

## 4. Gamma Fit Validation (Training Data)

**Does Gamma(α, β) actually match a player's non-zero score distribution?**

Each subplot:
- **Colored histogram** — actual non-zero weekly PPR scores (2022–2024)
- **Red curve** — `scipy.stats.gamma(a=α, scale=β).pdf(x)` — the fitted theoretical density

A good fit has the curve tracking the histogram mode and tail.  Systematic misfit would suggest Gamma is the wrong family for that position.

In [ ]:
showcase = []
for pos, n in [("WR", 2), ("RB", 2), ("QB", 1), ("TE", 1)]:
    showcase.append(
        params_train[(params_train["position"] == pos) & ~params_train["fallback_used"]]
        .nlargest(n, "exp_score")
    )
showcase = pd.concat(showcase).reset_index(drop=True)

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()

for ax, (_, row) in zip(axes, showcase.iterrows()):
    actual = weekly_train.loc[
        (weekly_train["player_id"] == row["player_id"]) & (weekly_train["ppr_score"] > 0),
        "ppr_score",
    ].values

    x   = np.linspace(0, actual.max() * 1.3, 300)
    pdf = scipy_gamma.pdf(x, a=row["alpha"], scale=row["beta"])

    ax.hist(actual, bins=15, density=True, alpha=0.55,
            color=POS_COLORS.get(row["position"], "gray"), label=f"Actual (n={len(actual)})")
    ax.plot(x, pdf, color="firebrick", lw=2,
            label=f"Gamma(\u03b1={row['alpha']:.1f}, \u03b2={row['beta']:.1f})")
    ax.set_title(f"{row['player_name']} ({row['position']})\n\u03c0={row['pi']:.2f}  \u03bc={row['mu']:.1f}")
    ax.set_xlabel("PPR score")
    ax.set_ylabel("Density")
    ax.legend(fontsize=8)

fig.suptitle("Gamma Fit vs. Actual Non-Zero Scores \u2014 2022\u20132024 Training Data",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("gamma_fit_validation.png", bbox_inches="tight")
plt.show()
print("Saved gamma_fit_validation.png")

## 5. Zero-Inflation by Position

π captures **week-to-week availability risk**.  Differences by position reflect real football dynamics:

- **QB** — starters almost always active → very low π
- **RB** — high injury rate + committee backfields → wider spread, higher median π
- **WR/TE** — target share is volatile; scheme changes produce 0-production games

In [ ]:
well_fit = params_train[
    params_train["position"].isin(SKILL_POS) & ~params_train["fallback_used"]
].copy()

fig, ax = plt.subplots(figsize=(9, 5))
for pos in SKILL_POS:
    grp = well_fit[well_fit["position"] == pos]["pi"]
    ax.hist(
        grp, bins=25, alpha=0.55, density=True,
        color=POS_COLORS[pos],
        label=f"{pos}  (n={len(grp)}, median={grp.median():.2f})",
    )

ax.set_xlabel("\u03c0 (zero-inflation probability)")
ax.set_ylabel("Density")
ax.set_title("Distribution of \u03c0 by Position \u2014 2022\u20132024\n(well-fit players only, n_nonzero \u2265 4)")
ax.legend()
plt.tight_layout()
plt.savefig("pi_by_position.png", bbox_inches="tight")
plt.show()

print("\nDescriptive stats for \u03c0 by position:")
print(well_fit.groupby("position")["pi"].describe().round(3))

## 6. 2025 Holdout Validation

Do parameters estimated on **2022–2024** accurately predict **2025** behavior?

- **Left** — predicted μ (mean non-zero score) vs. actual 2025 μ
- **Right** — predicted π vs. actual 2025 zero rate

Points on the diagonal are perfectly calibrated.  Expect positive correlation but scatter — a single season (max 17 games) is noisy, so ±3 pts error in μ is normal.  Systematic bias (most points above or below the diagonal) would indicate the model consistently over- or under-estimates one quantity.

Included: players with ≥ 4 non-zero weeks in 2025 (same quality bar as `fit.py`).

In [ ]:
common_ids = set(params_train["player_id"]) & set(weekly_hold["player_id"])

hold_stats = []
for pid, grp in weekly_hold[weekly_hold["player_id"].isin(common_ids)].groupby("player_id"):
    nonzero = grp.loc[grp["ppr_score"] > 0, "ppr_score"]
    n_poss  = 17 * grp["season"].nunique()
    if len(nonzero) >= 4:
        hold_stats.append({
            "player_id":      pid,
            "actual_mu_2025": float(nonzero.mean()),
            "actual_pi_2025": float(np.clip(1 - len(nonzero) / n_poss, 0, 1)),
        })

hold_df = pd.DataFrame(hold_stats)
val = (
    params_train[
        ~params_train["fallback_used"] & params_train["position"].isin(SKILL_POS)
    ].merge(hold_df, on="player_id")
)
print(f"Validation set: {len(val)} players")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: mu calibration
ax = axes[0]
for pos, grp in val.groupby("position"):
    ax.scatter(grp["mu"], grp["actual_mu_2025"], alpha=0.45, s=25,
               color=POS_COLORS.get(pos, "gray"), label=pos)
lim = max(val["mu"].max(), val["actual_mu_2025"].max()) * 1.05
ax.plot([0, lim], [0, lim], "k--", lw=1, label="Perfect calibration")
ax.set_xlabel("Predicted \u03bc (2022\u20132024 fit)")
ax.set_ylabel("Actual \u03bc (2025)")
ax.set_title("Mean Non-Zero Score: Predicted vs. Actual 2025")
ax.legend(fontsize=9)

# Right: pi calibration
ax = axes[1]
for pos, grp in val.groupby("position"):
    ax.scatter(grp["pi"], grp["actual_pi_2025"], alpha=0.45, s=25,
               color=POS_COLORS.get(pos, "gray"), label=pos)
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Perfect calibration")
ax.set_xlabel("Predicted \u03c0 (2022\u20132024 fit)")
ax.set_ylabel("Actual \u03c0 (2025)")
ax.set_title("Zero-Inflation: Predicted vs. Actual 2025")
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig("holdout_validation.png", bbox_inches="tight")
plt.show()

val["mu_err"] = val["actual_mu_2025"] - val["mu"]
val["pi_err"] = val["actual_pi_2025"] - val["pi"]
print("\nCalibration summary  (actual 2025 \u2212 predicted from 2022\u20132024):")
print(val[["mu_err", "pi_err"]].describe().round(3))

## 7. Refit on 2022–2025 (Production Parameters)

Refit on all four seasons to produce the best available estimates for 2026.  This call writes `data/player_params.parquet` — the file downstream code loads by default.

The table shows how α, β, π, and expected score shift for the top-10 players after adding 2025 data.

In [ ]:
params_prod = fit_player_params(weekly_all, cache=True)
params_prod["exp_score"] = (1 - params_prod["pi"]) * params_prod["mu"]

n_fb_prod = params_prod["fallback_used"].sum()
print(f"Production params: {len(params_prod):,} players  |  Fallback: {n_fb_prod:,} ({100*n_fb_prod/len(params_prod):.1f}%)\n")

top10_prod = (
    params_prod[params_prod["position"].isin(SKILL_POS) & ~params_prod["fallback_used"]]
    .nlargest(10, "exp_score")
    [["player_id", "player_name", "position", "alpha", "beta", "pi", "mu", "exp_score"]]
)

compare = top10_prod.merge(
    params_train[["player_id", "mu", "pi", "exp_score"]].rename(
        columns={"mu": "mu_22_24", "pi": "pi_22_24", "exp_score": "exp_22_24"}
    ),
    on="player_id", how="left",
)
compare["\u0394mu"]  = (compare["mu"]        - compare["mu_22_24"]).round(2)
compare["\u0394pi"]  = (compare["pi"]        - compare["pi_22_24"]).round(3)
compare["\u0394exp"] = (compare["exp_score"] - compare["exp_22_24"]).round(2)

compare[["player_name", "position", "alpha", "beta", "pi", "mu", "exp_score", "\u0394mu", "\u0394pi", "\u0394exp"]].round(3)

## 8. Simulation Sanity Check

Simulate 50,000 trials for a realistic 7-player starting lineup built from the top production parameters:
1 QB · 2 RB · 2 WR · 1 TE · 1 FLEX (best remaining RB or WR by expected score)

The resulting distribution should be roughly bell-shaped (Central Limit Theorem across 7 players) with a right tail from occasional multi-player boom weeks.  The P10–P90 range captures the realistic floor-to-ceiling for this lineup.

In [ ]:
lineup = (
    top_n_at_pos(params_prod, "QB", 1)
    + top_n_at_pos(params_prod, "RB", 2)
    + top_n_at_pos(params_prod, "WR", 2)
    + top_n_at_pos(params_prod, "TE", 1)
)
used_ids = {p["player_id"] for p in lineup}

flex_pool = [
    p for p in top_n_at_pos(params_prod, "RB", 5) + top_n_at_pos(params_prod, "WR", 5)
    if p["player_id"] not in used_ids
]
flex_pool.sort(
    key=lambda p: params_prod.loc[params_prod["player_id"] == p["player_id"], "exp_score"].values[0],
    reverse=True,
)
lineup.append(flex_pool[0])

print("Starting lineup:")
for p in lineup:
    row = params_prod[params_prod["player_id"] == p["player_id"]].iloc[0]
    print(f"  {p['player_name']:25s} {p['position']:2s}  mu={row['mu']:.1f}  pi={row['pi']:.2f}  E[score]={row['exp_score']:.1f}")

scores = simulate_team_score(lineup, params_prod, n=50_000, rng=42)

p10 = np.percentile(scores, 10)
p90 = np.percentile(scores, 90)

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(scores, bins=80, density=True, color="steelblue", alpha=0.7,
        edgecolor="white", linewidth=0.3)
ax.axvline(scores.mean(), color="firebrick", lw=2,   linestyle="--", label=f"Mean = {scores.mean():.1f}")
ax.axvline(p10,           color="orange",   lw=1.5, linestyle=":",  label=f"P10  = {p10:.1f}")
ax.axvline(p90,           color="green",    lw=1.5, linestyle=":",  label=f"P90  = {p90:.1f}")
ax.set_xlabel("Total PPR Score")
ax.set_ylabel("Density")
ax.set_title(f"Simulated Team Score Distribution  (n=50,000)\nMedian={np.median(scores):.1f}   Std={scores.std():.1f}")
ax.legend()
plt.tight_layout()
plt.savefig("sim_sanity.png", bbox_inches="tight")
plt.show()

## 9. Matchup Demo

Alternating-pick draft from the top 18 skill players (by expected score) splits them into two realistic rosters.  We simulate the matchup twice:

1. **No byes** — baseline win probability
2. **Team A's top pick on bye** — shows how much a single star's absence shifts the win probability

The overlap between the two score distributions directly determines P(win): more overlap → closer to 50/50.

In [ ]:
last_team = (
    weekly_all.sort_values(["season", "week"])
    .groupby("player_id")["recent_team"]
    .last()
    .reset_index()
    .rename(columns={"recent_team": "team"})
)

top18 = (
    params_prod[params_prod["position"].isin(SKILL_POS) & ~params_prod["fallback_used"]]
    .nlargest(18, "exp_score")
    .merge(last_team, on="player_id", how="left")
    .reset_index(drop=True)
)

roster_a = top18.iloc[0::2][["player_id", "player_name", "position", "team"]].to_dict("records")
roster_b = top18.iloc[1::2][["player_id", "player_name", "position", "team"]].to_dict("records")

print("Team A:", [p["player_name"] for p in roster_a])
print("Team B:", [p["player_name"] for p in roster_b])

result     = simulate_matchup(roster_a, roster_b, params_prod, n=50_000, rng=42)
sa, sb     = result["score_dist_a"], result["score_dist_b"]
bins       = np.linspace(min(sa.min(), sb.min()), max(sa.max(), sb.max()), 80)

bye_player = roster_a[0]
bye_team   = bye_player["team"]
print(f"\nBye scenario: {bye_player['player_name']} ({bye_team}) is on bye")

result_bye = simulate_matchup(
    roster_a, roster_b, params_prod, n=50_000, rng=42,
    bye_teams=[bye_team],
)
sa2, sb2 = result_bye["score_dist_a"], result_bye["score_dist_b"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (s_a, s_b, title) in zip(axes, [
    (sa,  sb,  f"No Byes  |  P(A wins) = {result['p_win_a']:.3f} \u00b1 {result['se']:.4f}"),
    (sa2, sb2, f"{bye_player['player_name']} on bye  |  P(A wins) = {result_bye['p_win_a']:.3f} \u00b1 {result_bye['se']:.4f}"),
]):
    ax.hist(s_a, bins=bins, density=True, alpha=0.55, color="steelblue", label=f"Team A  (mean={s_a.mean():.1f})")
    ax.hist(s_b, bins=bins, density=True, alpha=0.55, color="firebrick", label=f"Team B  (mean={s_b.mean():.1f})")
    ax.axvline(s_a.mean(), color="steelblue", lw=2, linestyle="--")
    ax.axvline(s_b.mean(), color="firebrick",  lw=2, linestyle="--")
    ax.set_xlabel("Total PPR Score")
    ax.set_ylabel("Density")
    ax.set_title(title)
    ax.legend()

plt.tight_layout()
plt.savefig("matchup_demo.png", bbox_inches="tight")
plt.show()

delta_pp = (result_bye["p_win_a"] - result["p_win_a"]) * 100
print(f"\nWin probability shift from bye: {result['p_win_a']:.3f} \u2192 {result_bye['p_win_a']:.3f}  ({delta_pp:+.1f} pp)")